# DuckDB + SLayer, in Python

DuckDB reads a **48 KB CSV straight off a CDN** over `httpfs`; SLayer auto-ingests its schema and answers queries about it. Nothing is copied locally — the DuckDB view points at the URL, and every query reaches back over the wire.

**Prerequisites:** `pip install motley-slayer` (DuckDB ships with it).

## 1. Point DuckDB at the file online

We create a **view** over the remote CSV inside a file-backed DuckDB database. DuckDB auto-installs the `httpfs` extension and streams the file; no rows land on disk. We close the connection right after — DuckDB won't share a read-write file across connections, and SLayer opens its own.

In [ ]:
import shutil
import warnings
from pathlib import Path

import duckdb
import pandas as pd

# duckdb-engine warns that it can't reflect indices — irrelevant here.
warnings.filterwarnings("ignore", message=".*reflection on indices.*")

from slayer.async_utils import run_sync
from slayer.client.slayer_client import SlayerClient
from slayer.core.models import DatasourceConfig
from slayer.engine.query_engine import SlayerQueryEngine
from slayer.storage.yaml_storage import YAMLStorage

CACHE = Path(".cache/python")
shutil.rmtree(CACHE, ignore_errors=True)
CACHE.mkdir(parents=True)

DB_PATH = (CACHE / "weather.duckdb").resolve()
CSV_URL = "https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv"

con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE VIEW weather AS SELECT * FROM '{CSV_URL}'")
n_rows = con.sql("SELECT count(*) FROM weather").fetchone()[0]
assert n_rows == 1461, f"expected 1461 rows, got {n_rows}"
schema = con.sql("DESCRIBE weather").fetchdf()
con.close()
schema

## 2. Auto-ingest the schema, SLayer-side

SLayer introspects the view — off the live URL — and builds a dataset named `weather`, one field per column with a datatype. No hand-written dataset.

In [ ]:
storage = YAMLStorage(base_dir=str(CACHE / "datasets"))
ds = DatasourceConfig(name="weather_db", type="duckdb", database=str(DB_PATH))
run_sync(storage.save_datasource(ds))

engine = SlayerQueryEngine(storage=storage)
engine.ingest_idempotent_sync(ds)

datasets = run_sync(storage.list_datasets(data_source="weather_db"))
assert "weather" in datasets, f"weather dataset not ingested; got {datasets}"

weather = run_sync(storage.get_dataset(name="weather", data_source="weather_db"))
pd.DataFrame([{"field": f.name, "datatype": f.datatype} for f in weather.fields])

## 3. A warm-up query

Average high and day count per weather type — a plain grouped aggregation, to confirm the dataset answers questions. Aggregations are chosen at query time (`avg(weather.temp_max)`, `count(weather.*)`).

In [ ]:
client = SlayerClient(storage=storage)

warmup = client.query_sync(
    {
        "population": "weather",
        "dimensions": ["weather.weather"],
        "metrics": [
            {"metric": "avg(weather.temp_max)", "alias": "avg_high"},
            {"metric": "count(weather.*)", "alias": "days"},
        ],
        "order": [{"field": "days", "direction": "desc"}],
    }
)
pd.DataFrame(warmup.data)

## 4. The hero query: an aggregate as a dimension, plus a ranking transform

Two query-time features at the month grain, in **one** single-stage query:

- **A dimension computed from an aggregate.** `season` — `CASE WHEN avg(weather.temp_max, partition_by=weather.date) >= 18 THEN 'warm' ELSE 'cool' END` — groups each month *warm* or *cool* by its own average high. That value only exists after aggregating, so SLayer computes it in a synthesized stage and regroups on it (the `partition_by=weather.date` grain follows the monthly time dimension).
- **A ranking transform in a metric.** `rank(sum(weather.precipitation), direction='desc')` orders the months by rainfall, with 1 the wettest.

Ordered wettest-first, the result tells a story: every one of Seattle's rainiest months is *cool*-season.

In [ ]:
hero = {
    "population": "weather",
    "time_dimensions": [{"field": "weather.date", "granularity": "month"}],
    "dimensions": [
        {
            "field": "CASE WHEN avg(weather.temp_max, partition_by=weather.date) >= 18 THEN 'warm' ELSE 'cool' END",
            "alias": "season",
        }
    ],
    "metrics": [
        {"metric": "sum(weather.precipitation)", "alias": "total_rain"},
        {"metric": "rank(sum(weather.precipitation), direction='desc')", "alias": "rain_rank"},
    ],
    "order": [{"field": "rain_rank", "direction": "asc"}],
}

result = client.query_sync(hero)
df = pd.DataFrame(result.data)

assert len(df) == 48, f"expected 48 monthly rows, got {len(df)}"
# ordered wettest-first: the rainiest month is cool-season
assert df.iloc[0]["season"] == "cool", "expected the wettest month to be cool-season"
df

## 5. The SQL SLayer ran

Why use SLayer at all, instead of writing SQL directly? Well, here's the SQL corresponding to the above query. Which one do you think is easier for agents to write, or for humans to audit?

In [ ]:
assert result.sql, "expected generated SQL on the response"
print(result.sql)

---

That's a semantic layer over a file on the internet, in a page of code. See the [command-line version](duckdb_cli_nb.ipynb) for the same demo driven entirely through the `slayer` CLI, the [aggregations](../07_aggregations/aggregations.md) guide for computing dimensions from aggregates, and [expressions](../../concepts/expressions.md) for the full transform vocabulary.